In [1]:
import pandas as pd
import numpy as np
import os
import json
import warnings
warnings.filterwarnings('ignore')

BASE = '/kaggle/input/datasets/bhanuteja2007/aqi-data'   # ⚠️ update if Kaggle shows a different path

station_files = {
    'Bollaram'          : f'{BASE}/bollaram.csv',
    'Central_University': f'{BASE}/central_university.csv',
    'Zoo_Park'          : f'{BASE}/zoo_park.csv',
    'ICRISAT_Patancheru': f'{BASE}/icrisat_patancheru.csv',
    'IDA_Pashamylaram'  : f'{BASE}/ida_pashamylaram.csv',
}

dfs = []
for station_name, filepath in station_files.items():
    df_temp = pd.read_csv(filepath)
    df_temp['Station'] = station_name
    dfs.append(df_temp)

df_raw = pd.concat(dfs, ignore_index=True)
print(f"✅ Combined shape: {df_raw.shape}")

✅ Combined shape: (14644, 8)


In [2]:
df_raw.columns = df_raw.columns.str.strip()
df_raw = df_raw.rename(columns={
    'date': 'Date', 'pm25': 'PM25', 'pm10': 'PM10',
    'o3': 'O3', 'no2': 'NO2', 'so2': 'SO2', 'co': 'CO',
})

df_raw['Date'] = pd.to_datetime(df_raw['Date'], format='%Y/%m/%d', errors='coerce')
df_raw = df_raw.dropna(subset=['Date'])

pollutant_cols = ['PM25', 'PM10', 'O3', 'NO2', 'SO2', 'CO']
for col in pollutant_cols:
    if col in df_raw.columns:
        df_raw[col] = pd.to_numeric(df_raw[col].astype(str).str.strip(), errors='coerce')

df_raw = df_raw.sort_values(['Station', 'Date']).reset_index(drop=True)

def fill_station(group):
    for col in pollutant_cols:
        if col in group.columns:
            group[col] = group[col].interpolate(method='linear').fillna(method='ffill').fillna(method='bfill')
    return group

df_clean = df_raw.groupby('Station', group_keys=False).apply(fill_station)
print("✅ Cleaning done. Missing values:")
print(df_clean[pollutant_cols].isnull().sum())

✅ Cleaning done. Missing values:
PM25    0
PM10    0
O3      0
NO2     0
SO2     0
CO      0
dtype: int64


In [3]:
def sub_index_pm25(val):
    if pd.isna(val): return np.nan
    bp = [(0,30,0,50),(30,60,51,100),(60,90,101,200),(90,120,201,300),(120,250,301,400),(250,500,401,500)]
    for lo,hi,alo,ahi in bp:
        if lo <= val <= hi: return round(((ahi-alo)/(hi-lo))*(val-lo)+alo, 1)
    return 500.0

def sub_index_pm10(val):
    if pd.isna(val): return np.nan
    bp = [(0,50,0,50),(50,100,51,100),(100,250,101,200),(250,350,201,300),(350,430,301,400),(430,600,401,500)]
    for lo,hi,alo,ahi in bp:
        if lo <= val <= hi: return round(((ahi-alo)/(hi-lo))*(val-lo)+alo, 1)
    return 500.0

def sub_index_no2(val):
    if pd.isna(val): return np.nan
    bp = [(0,40,0,50),(40,80,51,100),(80,180,101,200),(180,280,201,300),(280,400,301,400),(400,800,401,500)]
    for lo,hi,alo,ahi in bp:
        if lo <= val <= hi: return round(((ahi-alo)/(hi-lo))*(val-lo)+alo, 1)
    return 500.0

def sub_index_so2(val):
    if pd.isna(val): return np.nan
    bp = [(0,40,0,50),(40,80,51,100),(80,380,101,200),(380,800,201,300),(800,1600,301,400),(1600,2100,401,500)]
    for lo,hi,alo,ahi in bp:
        if lo <= val <= hi: return round(((ahi-alo)/(hi-lo))*(val-lo)+alo, 1)
    return 500.0

def sub_index_o3(val):
    if pd.isna(val): return np.nan
    bp = [(0,50,0,50),(50,100,51,100),(100,168,101,200),(168,208,201,300),(208,748,301,400),(748,1000,401,500)]
    for lo,hi,alo,ahi in bp:
        if lo <= val <= hi: return round(((ahi-alo)/(hi-lo))*(val-lo)+alo, 1)
    return 500.0

def sub_index_co(val):
    if pd.isna(val): return np.nan
    bp = [(0,1,0,50),(1,2,51,100),(2,10,101,200),(10,17,201,300),(17,34,301,400),(34,50,401,500)]
    for lo,hi,alo,ahi in bp:
        if lo <= val <= hi: return round(((ahi-alo)/(hi-lo))*(val-lo)+alo, 1)
    return 500.0

df_clean['SI_PM25'] = df_clean['PM25'].apply(sub_index_pm25)
df_clean['SI_PM10'] = df_clean['PM10'].apply(sub_index_pm10)
df_clean['SI_NO2']  = df_clean['NO2'].apply(sub_index_no2)
df_clean['SI_SO2']  = df_clean['SO2'].apply(sub_index_so2)
df_clean['SI_O3']   = df_clean['O3'].apply(sub_index_o3)
df_clean['SI_CO']   = df_clean['CO'].apply(sub_index_co)

si_cols = ['SI_PM25','SI_PM10','SI_NO2','SI_SO2','SI_O3','SI_CO']
df_clean['AQI'] = df_clean[si_cols].max(axis=1)

def aqi_category(val):
    if pd.isna(val): return 'Unknown'
    if val <= 50: return 'Good'
    if val <= 100: return 'Satisfactory'
    if val <= 200: return 'Moderate'
    if val <= 300: return 'Poor'
    if val <= 400: return 'Very Poor'
    return 'Severe'

df_clean['AQI_Category'] = df_clean['AQI'].apply(aqi_category)
print("✅ AQI calculated!")

✅ AQI calculated!


In [4]:
from sklearn.preprocessing import LabelEncoder

df_clean['DayOfWeek'] = df_clean['Date'].dt.dayofweek
df_clean['Month'] = df_clean['Date'].dt.month
df_clean['Year'] = df_clean['Date'].dt.year
df_clean['IsWeekend'] = (df_clean['DayOfWeek'] >= 5).astype(int)

def get_season(m):
    if m in [12,1,2]: return 0
    if m in [3,4,5]: return 1
    if m in [6,7,8,9]: return 2
    return 3

df_clean['Season'] = df_clean['Month'].apply(get_season)

le = LabelEncoder()
df_clean['Station_enc'] = le.fit_transform(df_clean['Station'])

def add_lags(group):
    for lag in [1, 2, 3, 7, 14, 30]:
        group[f'AQI_lag_{lag}d'] = group['AQI'].shift(lag)
    group['AQI_roll_mean_7d'] = group['AQI'].rolling(7).mean()
    group['AQI_roll_std_7d'] = group['AQI'].rolling(7).std()
    group['AQI_roll_max_7d'] = group['AQI'].rolling(7).max()
    group['AQI_roll_mean_30d'] = group['AQI'].rolling(30).mean()
    return group

df_feat = df_clean.groupby('Station', group_keys=False).apply(add_lags)
df_model = df_feat.dropna().reset_index(drop=True)
print("✅ Features added! Final shape:", df_model.shape)

✅ Features added! Final shape: (14494, 32)


In [5]:
from sklearn.preprocessing import MinMaxScaler

feature_cols = [
    'PM25','PM10','NO2','SO2','O3','CO','Month','DayOfWeek','Season','IsWeekend',
    'Station_enc','AQI_lag_1d','AQI_lag_2d','AQI_lag_3d','AQI_lag_7d','AQI_lag_14d',
    'AQI_roll_mean_7d','AQI_roll_std_7d','AQI_roll_max_7d','AQI_roll_mean_30d'
]
TARGET = 'AQI'
LOOKBACK = 30
HORIZON = 7

sc_X = MinMaxScaler()
sc_y = MinMaxScaler()

df_model_scaled = df_model.copy()
df_model_scaled[feature_cols] = sc_X.fit_transform(df_model[feature_cols])
df_model_scaled['AQI_scaled'] = sc_y.fit_transform(df_model[[TARGET]])

print("✅ Scaling done.")

✅ Scaling done.


In [6]:
X_train_list, y_train_list = [], []
X_test_list, y_test_list = [], []
test_meta = []

for station in df_model_scaled['Station'].unique():
    df_st = df_model_scaled[df_model_scaled['Station'] == station].sort_values('Date').reset_index(drop=True)
    df_st_raw = df_model[df_model['Station'] == station].sort_values('Date').reset_index(drop=True)

    Xf = df_st[feature_cols].values
    yf = df_st['AQI_scaled'].values.reshape(-1, 1)

    Xs, ys, meta_rows = [], [], []
    for i in range(LOOKBACK, len(df_st) - HORIZON):
        Xs.append(Xf[i - LOOKBACK:i])
        ys.append(yf[i:i + HORIZON].flatten())
        meta_rows.append(df_st_raw.iloc[i - 1])

    Xs, ys = np.array(Xs), np.array(ys)
    n = len(Xs)
    split = int(0.8 * n)

    X_train_list.append(Xs[:split]); y_train_list.append(ys[:split])
    X_test_list.append(Xs[split:]);  y_test_list.append(ys[split:])
    test_meta.extend(meta_rows[split:])

X_tr = np.concatenate(X_train_list, axis=0)
y_tr = np.concatenate(y_train_list, axis=0)
X_te = np.concatenate(X_test_list, axis=0)
y_te = np.concatenate(y_test_list, axis=0)

print(f"✅ Train: {X_tr.shape} | Test: {X_te.shape} | Test metadata rows: {len(test_meta)}")
assert len(test_meta) == len(X_te), "Mismatch — metadata must align with test set!"

✅ Train: (11446, 30, 20) | Test: (2863, 30, 20) | Test metadata rows: 2863


In [7]:
from sklearn.multioutput import MultiOutputRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score

X_flat_tr = X_tr.reshape(len(X_tr), -1)
X_flat_te = X_te.reshape(len(X_te), -1)

xgb_model = MultiOutputRegressor(
    XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.05, random_state=42, n_jobs=-1),
    n_jobs=-1
)
xgb_model.fit(X_flat_tr, y_tr)
xgb_pred_scaled = xgb_model.predict(X_flat_te)

y_true = sc_y.inverse_transform(y_te)
y_pred = sc_y.inverse_transform(xgb_pred_scaled)
rmse = np.sqrt(mean_squared_error(y_true.flatten(), y_pred.flatten()))
r2 = r2_score(y_true.flatten(), y_pred.flatten())
print(f"✅ XGBoost done! RMSE={rmse:.2f} | R²={r2:.4f}")

✅ XGBoost done! RMSE=52.65 | R²=0.6295


In [8]:
xgb_pred = sc_y.inverse_transform(xgb_pred_scaled)

si_to_pollutant = {
    'SI_PM25': 'PM2.5', 'SI_PM10': 'PM10', 'SI_NO2': 'NO2',
    'SI_SO2': 'SO2', 'SI_O3': 'O3', 'SI_CO': 'CO'
}

def get_dominant_pollutant(row):
    best_col = max(si_to_pollutant.keys(), key=lambda c: row[c] if pd.notna(row[c]) else -1)
    return si_to_pollutant[best_col]

records = []
for k in range(len(xgb_pred)):
    meta_row = test_meta[k]
    day1_aqi = xgb_pred[k][0]
    day7_aqi = xgb_pred[k][-1]
    diff = day7_aqi - day1_aqi
    trend = "rising" if diff > 5 else ("falling" if diff < -5 else "stable")

    records.append({
        "station": meta_row['Station'],
        "date": meta_row['Date'],
        "predicted_aqi": round(float(day1_aqi), 1),
        "dominant_pollutant": get_dominant_pollutant(meta_row),
        "trend": trend,
    })

pred_df = pd.DataFrame(records)
latest_per_station = pred_df.sort_values("date").groupby("station").tail(1).reset_index(drop=True)
print(latest_per_station)

              station       date  predicted_aqi dominant_pollutant    trend
0            Bollaram 2026-02-16          317.8              PM2.5   stable
1    IDA_Pashamylaram 2026-05-15          178.6              PM2.5   stable
2  ICRISAT_Patancheru 2026-05-16          175.8              PM2.5   rising
3  Central_University 2026-05-16          167.7              PM2.5   rising
4            Zoo_Park 2026-05-16          234.6              PM2.5  falling


In [9]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
os.environ["GEMINI_API_KEY"] = user_secrets.get_secret("GEMINI_API_KEY")

!pip uninstall -y google-generativeai
!pip install -q -U google-genai

from google import genai

client = genai.Client()
MODEL_NAME = "gemini-2.5-flash"

ADVISORY_PROMPT_TEMPLATE = """You are a public health communication assistant for
a city air quality monitoring system in Hyderabad, India.

Given the following AQI forecast for one monitoring station, write a SHORT,
clear public advisory (3-4 sentences) that a general resident (not a
scientist) can understand and act on. Mention the station/area name, the
expected air quality category, the dominant pollutant in plain terms, and
one concrete piece of advice for sensitive groups (children, elderly,
asthma/respiratory patients).

Then provide the SAME advisory translated naturally into conversational
Telugu (not a literal word-for-word translation).

Station: {station}
Predicted AQI: {aqi} ({category})
Dominant pollutant: {pollutant}
Trend: {trend}
Forecast window: {forecast_window}

Respond ONLY in this exact JSON format, no markdown fences, no extra text:
{{
  "advisory_en": "...",
  "advisory_te": "..."
}}
"""

def aqi_to_category(aqi):
    if aqi <= 50: return "Good"
    elif aqi <= 100: return "Satisfactory"
    elif aqi <= 200: return "Moderate"
    elif aqi <= 300: return "Poor"
    elif aqi <= 400: return "Very Poor"
    return "Severe"

def generate_advisory(station, predicted_aqi, dominant_pollutant, trend="rising", forecast_window="next 24 hours"):
    category = aqi_to_category(predicted_aqi)
    prompt = ADVISORY_PROMPT_TEMPLATE.format(
        station=station, aqi=predicted_aqi, category=category,
        pollutant=dominant_pollutant, trend=trend, forecast_window=forecast_window,
    )
    response = client.models.generate_content(model=MODEL_NAME, contents=prompt)
    raw_text = response.text.strip().replace("```json", "").replace("```", "").strip()
    try:
        parsed = json.loads(raw_text)
    except json.JSONDecodeError:
        parsed = {"advisory_en": raw_text, "advisory_te": ""}
    return {
        "station": station, "predicted_aqi": predicted_aqi, "category": category,
        "dominant_pollutant": dominant_pollutant,
        "advisory_en": parsed.get("advisory_en", ""),
        "advisory_te": parsed.get("advisory_te", ""),
    }

print("✅ Gemini function ready.")

Found existing installation: google-generativeai 0.8.6
Uninstalling google-generativeai-0.8.6:
  Successfully uninstalled google-generativeai-0.8.6
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.5/53.5 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 958.0/958.0 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.3/252.3 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.3/472.3 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 71.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-cola

In [10]:
all_advisories = []
for row in latest_per_station.itertuples():
    result = generate_advisory(
        station=row.station,
        predicted_aqi=row.predicted_aqi,
        dominant_pollutant=row.dominant_pollutant,
        trend=row.trend,
        forecast_window="next 24 hours",
    )
    all_advisories.append(result)
    print(result["station"], "→", result["category"])
    print(result["advisory_en"])
    print(result["advisory_te"])
    print("---")

with open("/kaggle/working/hyderabad_aqi_advisories.json", "w", encoding="utf-8") as f:
    json.dump(all_advisories, f, indent=2, ensure_ascii=False)

print("Saved", len(all_advisories), "advisories")

Bollaram → Very Poor
Attention Bollaram residents: Air quality is forecast to be 'Very Poor' for the next 24 hours. The main concern is tiny particles (PM2.5) that can deeply penetrate your lungs. Children, the elderly, and individuals with respiratory conditions like asthma should minimize extended outdoor activities. If you must go out, consider wearing a mask.
బొల్లారం నివాసులకు ముఖ్య గమనిక: రాబోయే 24 గంటల్లో గాలి నాణ్యత 'చాలా అధ్వాన్నంగా' ఉండబోతోంది. ముఖ్యంగా, ఊపిరితిత్తుల్లోకి సులభంగా చేరే చిన్న ధూళి కణాలు (PM2.5) అధికంగా ఉన్నాయి. పిల్లలు, వృద్ధులు మరియు ఆస్తమా వంటి శ్వాసకోశ సమస్యలు ఉన్నవారు ఎక్కువసేపు బయట ఉండకుండా చూసుకోవాలి. అత్యవసరమైతే మాస్కులు ధరించడం మంచిది.
---
IDA_Pashamylaram → Moderate
Air quality in IDA_Pashamylaram is forecast to be Moderate over the next 24 hours. This means the presence of fine particulate matter (PM2.5) in the air, which can be concerning. Children, the elderly, and those with respiratory issues should limit prolonged outdoor exertion during this per